# Embedding & Storage (Chroma DB)

Takes Niv's chunks (`data/chunks.json`), embeds them with **`BAAI/bge-small-en-v1.5`** (sentence-transformers), and stores them in a persistent Chroma collection (`data/chroma_db`).

1. Why this model
2. Check the model fits our chunks (truncation)
3. Retrieval check on probe queries
4. Build the Chroma collection (`src/vector_store.py`)
5. Sanity-check queries and metadata filtering

The reusable code lives in `src/vector_store.py` so the testing step can import `query()` directly.

## 1. Why `bge-small-en-v1.5`

We compared four sentence-transformers models on our 918 chunks, using the 20 probe queries in section 3 (CPU timings):

| Model | Dim | Max tokens | % chunks truncated | hit@1 | hit@5 | MRR | Embed time |
|---|---|---|---|---|---|---|---|
| `all-MiniLM-L6-v2` | 384 | 256 | **34.5%** | 0.95 | 1.0 | 0.975 | 17 s |
| `all-mpnet-base-v2` | 768 | 384 | 3.6% | 0.90 | 1.0 | 0.938 | 351 s |
| **`BAAI/bge-small-en-v1.5`** | 384 | 512 | **0.2%** | **0.95** | 1.0 | **0.975** | 156 s |
| `BAAI/bge-base-en-v1.5` | 768 | 512 | 0.2% | 0.90 | 1.0 | 0.950 | 424 s |

- **Retrieval scores are effectively tied**: every model finds the right paper in the top 5 for all 20 queries, and 0.95 vs 0.90 hit@1 is a single query. They can't separate the models on their own.
- **MiniLM truncates a third of our chunks**: chunks have a median of ~232 tokens but MiniLM stops reading at 256, so the end of 34.5% of chunks is never embedded.
- **bge-small vs bge-base**: both are trained for query-to-passage retrieval and read 512 tokens. The base model is ~3x slower with 2x larger vectors and scored no better here, so the small one is the better fit for a free Colab runtime.
- **Local model, not an API**: the project plan keeps embeddings local (sentence-transformers + ChromaDB) and reserves the Gemini API quota for answer generation.

Caveat: these probes check the *paper* only. The page-level check with the ground-truth Q&A pairs may still justify moving to `bge-base-en-v1.5`, which is a one-line change of `MODEL_NAME` in `src/vector_store.py`.

In [1]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("../src").resolve()))
import vector_store as vs

print("Embedding model:", vs.MODEL_NAME)

C:\Users\tanis\OneDrive\Desktop\Zayi\Zayi-1A-agentic-internal-knowledge-system-for-r-and-d\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Embedding model: BAAI/bge-small-en-v1.5


In [2]:
chunks = vs.load_chunks()
texts = [c["page_content"] for c in chunks]
paper_ids = np.array([c["metadata"]["paper_id"] for c in chunks])
print(f"Loaded {len(chunks)} chunks from {len(set(paper_ids))} papers")

Loaded 918 chunks from 10 papers


## 2. Does the model fit our chunks?

In [3]:
model = vs.get_model()
lengths = np.array([len(ids) for ids in model.tokenizer(texts)["input_ids"]])

pd.Series({
    "embedding dim": model.get_sentence_embedding_dimension(),
    "max_seq_length": model.max_seq_length,
    "median chunk tokens": int(np.median(lengths)),
    "longest chunk tokens": int(lengths.max()),
    "chunks truncated": int((lengths > model.max_seq_length).sum()),
})

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2488.73it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (513 > 512). Running this sequence through the model will result in indexing errors


C:\Users\tanis\AppData\Local\Temp\ipykernel_31416\3687684109.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  "embedding dim": model.get_sentence_embedding_dimension(),


embedding dim           384
max_seq_length          512
median chunk tokens     232
longest chunk tokens    516
chunks truncated          2
dtype: int64

## 3. Retrieval check on probe queries

Two hand-written questions per paper, each labelled with the paper that answers it. This is a paper-level check (did we retrieve chunks from the right paper?), a stand-in until the chunk-level ground-truth Q&A pairs are available.

- **hit@1**: top chunk comes from the expected paper
- **hit@5**: any of the top 5 chunks comes from the expected paper
- **MRR**: mean of 1 / rank of the first chunk from the expected paper

In [4]:
PROBES = [
    ("How does RAG combine parametric and non-parametric memory?", "2005.11401"),
    ("What is the difference between RAG-Sequence and RAG-Token models?", "2005.11401"),
    ("What are Naive RAG, Advanced RAG and Modular RAG paradigms?", "2312.10997"),
    ("What are the main challenges and future directions for retrieval-augmented LLMs?", "2312.10997"),
    ("What reflection tokens does the model generate to critique its own output?", "2310.11511"),
    ("How is the critic model trained to predict whether retrieval is needed?", "2310.11511"),
    ("What happens when the retrieval evaluator judges the retrieved documents as incorrect?", "2401.15884"),
    ("How does the decompose-then-recompose algorithm refine retrieved documents?", "2401.15884"),
    ("How can a language model interleave reasoning traces with actions on a Wikipedia API?", "2210.03629"),
    ("How does prompting with thoughts and actions perform on ALFWorld and WebShop?", "2210.03629"),
    ("Why do language models perform worse when relevant information is in the middle of a long input?", "2307.03172"),
    ("U-shaped performance curve depending on position of the answer document", "2307.03172"),
    ("How are faithfulness, answer relevance and context relevance measured automatically?", "2309.15217"),
    ("Reference-free evaluation framework for retrieval augmented generation pipelines", "2309.15217"),
    ("Why does low-rank adaptation add no additional inference latency?", "2106.09685"),
    ("Which attention weight matrices should the low-rank update be applied to?", "2106.09685"),
    ("How does tiling reduce memory reads and writes between GPU HBM and SRAM?", "2205.14135"),
    ("What is the IO complexity of block-sparse attention?", "2205.14135"),
    ("How are in-batch negatives used to train a dual encoder for passage retrieval?", "2004.04906"),
    ("How does a dense retriever compare to BM25 on top-20 retrieval accuracy?", "2004.04906"),
]
queries = [q for q, _ in PROBES]
expected = np.array([p for _, p in PROBES])

In [5]:
t0 = time.perf_counter()
doc_emb = model.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
print(f"Embedded {len(texts)} chunks in {time.perf_counter() - t0:.1f}s")

q_emb = model.encode([vs.QUERY_PREFIX + q for q in queries], normalize_embeddings=True)
ranked = np.argsort(-(q_emb @ doc_emb.T), axis=1)  # cosine similarity (normalized)
correct = paper_ids[ranked] == expected[:, None]
first_hit = np.where(correct.any(axis=1), correct.argmax(axis=1) + 1, np.inf)

print(f"hit@1: {correct[:, 0].mean():.2f}")
print(f"hit@5: {correct[:, :5].any(axis=1).mean():.2f}")
print(f"MRR:   {(1 / first_hit).mean():.3f}")

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Batches:   7%|▋         | 1/15 [00:04<00:58,  4.14s/it]

Batches:  13%|█▎        | 2/15 [00:06<00:41,  3.18s/it]

Batches:  20%|██        | 3/15 [00:09<00:37,  3.09s/it]

Batches:  27%|██▋       | 4/15 [00:12<00:32,  3.00s/it]

...

Batches:  87%|████████▋ | 13/15 [00:42<00:06,  3.00s/it]

Batches:  93%|█████████▎| 14/15 [00:44<00:02,  2.82s/it]

Batches: 100%|██████████| 15/15 [00:45<00:00,  2.23s/it]

Batches: 100%|██████████| 15/15 [00:45<00:00,  3.02s/it]

Embedded 918 chunks in 45.3s
hit@1: 0.95
hit@5: 1.00
MRR:   0.975


In [6]:
# Which queries missed at rank 1, and which paper won instead
for i in np.where(~correct[:, 0])[0]:
    print(f"- {queries[i]}")
    print(f"    expected {expected[i]}, got {paper_ids[ranked[i, 0]]} (right paper first at rank {int(first_hit[i])})")

- What are the main challenges and future directions for retrieval-augmented LLMs?
    expected 2312.10997, got 2309.15217 (right paper first at rank 2)


## 4. Build the Chroma collection

Storage settings:
- **Cosine distance** (`hnsw:space = cosine`) with normalized embeddings
- **IDs** `"<paper_id>_<chunk_index>"`, e.g. `2005.11401_0007`, so each chunk is stable and traceable
- **Metadata** keeps all of Niv's fields (`paper_id`, `title`, `category`, `pdf_url`, `total_pages`, `start_index`, `page_number`, `end_page`) plus `chunk_index` and `embedding_model`, so results can be cited back to a page and filtered by paper or category

In [7]:
collection = vs.build_collection()
print(f"Stored {collection.count()} chunks in collection '{vs.COLLECTION_NAME}' at {vs.DB_PATH}")

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Batches:   7%|▋         | 1/15 [00:03<00:49,  3.55s/it]

Batches:  13%|█▎        | 2/15 [00:06<00:39,  3.02s/it]

Batches:  20%|██        | 3/15 [00:09<00:38,  3.17s/it]

Batches:  27%|██▋       | 4/15 [00:12<00:34,  3.15s/it]

...

Batches:  87%|████████▋ | 13/15 [00:42<00:06,  3.08s/it]

Batches:  93%|█████████▎| 14/15 [00:45<00:02,  2.88s/it]

Batches: 100%|██████████| 15/15 [00:46<00:00,  2.29s/it]

Batches: 100%|██████████| 15/15 [00:46<00:00,  3.08s/it]

Stored 918 chunks in collection 'papers' at C:\Users\tanis\OneDrive\Desktop\Zayi\Zayi-1A-agentic-internal-knowledge-system-for-r-and-d\data\chroma_db


In [8]:
# Reopen from disk to confirm persistence
reloaded = vs.get_collection()
assert reloaded.count() == len(chunks)
assert reloaded.metadata["embedding_model"] == vs.MODEL_NAME
reloaded.get(ids=["2005.11401_0000"], include=["metadatas"])["metadatas"][0]

{'embedding_model': 'BAAI/bge-small-en-v1.5',
 'page_number': 1,
 'start_index': 0,
 'paper_id': '2005.11401',
 'end_page': 1,
 'chunk_index': 0,
 'pdf_url': 'https://arxiv.org/pdf/2005.11401.pdf',
 'category': 'RAG',
 'title': 'Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks',
 'total_pages': 19}

## 5. Query the store

In [9]:
def show(results):
    for r in results:
        m = r["metadata"]
        print(f"[{r['distance']:.3f}] {r['id']}  {m['title']}  (p.{m['page_number']}-{m['end_page']})")
        print("    " + r["text"][:250].replace("\n", " ") + "...\n")

show(vs.query("How does Self-RAG decide when to retrieve?", k=3))

[0.196] 2310.11511_0004  Self-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection  (p.1-2)
    given an input prompt and preceding generations, SELF -RAG first determines if augmenting the continued generation with retrieved passages would be helpful. If so, it outputs a retrieval token that calls a retriever model on demand (Step 1). Subseque...

[0.231] 2310.11511_0026  Self-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection  (p.6-6)
    the use of a soft re-ranking mechanism or hard constraints at inference (discussed next). 3.3 S ELF -RAG INFERENCE Generating reflection tokens to self-evaluate its own output makes SELF -RAG controllable during the inference phase, enabling it to ta...

[0.262] 2312.10997_0019  Retrieval-Augmented Generation for Large Language Models: A Survey  (p.5-5)
    retrievers through few-shot query generation [20], [21] .This comprehensive approach not only streamlines the retrieval process but also significantly 

In [10]:
show(vs.query("What is LoRA and why does it reduce the number of trainable parameters?", k=3))

[0.216] 2106.09685_0001  LoRA: Low-Rank Adaptation of Large Language Models  (p.1-1)
    layer of the Transformer architecture, greatly reducing the number of trainable parameters for downstream tasks. Compared to GPT-3 175B fine-tuned with Adam, LoRA can reduce the number of trainable parameters by 10,000 times and the GPU memory requir...

[0.246] 2106.09685_0005  LoRA: Low-Rank Adaptation of Large Language Models  (p.2-2)
    LoRA possesses several key advantages.  A pre-trained model can be shared and used to build many small LoRA modules for different tasks. We can freeze the shared model and efficiently switch tasks by replacing the matricesA andB in Figure 1, reducing...

[0.249] 2106.09685_0080  LoRA: Low-Rank Adaptation of Large Language Models  (p.21-21)
    Optimizer AdamW Batch Size 128 # Epoch 2 Warmup Tokens 250,000 LR Schedule Linear Learning Rate 5.00E-06 5.00E-04 1.00E-04 1.6E-03 1.00E-04 2.00E-04 Table 12: The training hyperparameters used for different GPT-3 adaption

In [11]:
# Metadata filter: restrict the search to one category
show(vs.query("How is retrieval quality evaluated?", k=3, where={"category": "Evaluation"}))

[0.237] 2309.15217_0003  RAGAS: Automated Evaluation of Retrieval Augmented Generation  (p.1-1)
    retrieval-augmented language modelling (Khandelwal et al., 2020; Borgeaud et al., 2022), recent work has suggested that simply adding retrieved documents to the input of a standard LM can also work well (Khattab et al., 2022; Ram et al., 2023; Shi et...

[0.245] 2309.15217_0022  RAGAS: Automated Evaluation of Retrieval Augmented Generation  (p.5-5)
    rank each answer based on Answer Relevancy. question: [question] answer 1: [answer 1] answer 2: [answer 2] The results in Table 1 show that our proposed metrics are much closer aligned with the human judgements than the predictions from the two basel...

[0.256] 2309.15217_0000  RAGAS: Automated Evaluation of Retrieval Augmented Generation  (p.1-1)
    Ragas: Automated Evaluation of Retrieval Augmented Generation Shahul Es , Jithin James , Luis Espinosa-Anke , Steven Schockaert   Exploding Gradients  CardiffNLP, Cardiff University, United 